##### Imports

In [1]:
import pandas as pd
from pathlib import Path

##### Load Dataset

In [2]:
BASE_DIR = Path("../dataset/annotations")

train_df = pd.read_csv(BASE_DIR / "EPIC_100_train.csv")
val_df = pd.read_csv(BASE_DIR / "EPIC_100_validation.csv")

print(f"Training clips   : {len(train_df):,}")
print(f"Validation clips : {len(val_df):,}")

Training clips   : 67,217
Validation clips : 9,668


##### Combine Train + Validation

In [3]:
df = pd.concat([train_df, val_df], ignore_index=True)

print(f"Total clips: {len(df):,}")

Total clips: 76,885


##### Keep only useful columns

In [4]:
df = df[
    [
        "narration_id",
        "participant_id",
        "video_id",
        "start_timestamp",
        "stop_timestamp",
        "narration",
        "verb",
        "noun",
    ]
]

df.head()

,narration_id,participant_id,video_id,start_timestamp,stop_timestamp,narration,verb,noun
0,P01_01_0,P01,P01_01,00:00.1,00:03.4,open door,open,door
1,P01_01_1,P01,P01_01,00:04.4,00:06.2,turn on light,turn-on,light
2,P01_01_10,P01,P01_01,00:25.0,00:26.2,open drawer,open,drawer
3,P01_01_100,P01,P01_01,07:59.8,08:00.9,take cup,take,cup
4,P01_01_101,P01,P01_01,08:01.5,08:02.2,open cupboard,open,cupboard


##### Search Functions

In [5]:
def search_action(verb=None, noun=None):
    """
    Search clips by verb and/or noun.
    """

    result = df.copy()

    if verb:
        result = result[
            result["verb"].str.lower() == verb.lower()
        ]

    if noun:
        result = result[
            result["noun"].str.lower() == noun.lower()
        ]

    return result.sort_values(
        ["participant_id", "video_id"]
    ).reset_index(drop=True)

In [6]:
search_action("cut", "onion")

,narration_id,participant_id,video_id,start_timestamp,stop_timestamp,narration,verb,noun
0,P01_01_82,P01,P01_01,06:32.6,06:38.8,cut onion,cut,onion
1,P01_01_83,P01,P01_01,06:35.4,06:47.0,still cutting onion,cut,onion
2,P01_01_84,P01,P01_01,06:48.4,07:02.2,still cutting onion,cut,onion
3,P01_09_227,P01,P01_09,18:03.9,18:10.6,cut onion,cut,onion
4,P01_09_229,P01,P01_09,18:14.1,18:16.8,cut onion,cut,onion
...,...,...,...,...,...,...,...,...
119,P26_35_27,P26,P26_35,00:02:02.44,00:02:11.94,cut onion,cut,onion
120,P28_04_75,P28,P28_04,06:31.1,06:41.8,cut onion,cut,onion
121,P28_16_1,P28,P28_16,00:00:04.01,00:00:10.20,cut onion,cut,onion
122,P28_16_25,P28,P28_16,00:01:22.12,00:01:25.80,cut onion,cut,onion


##### Narration Search

In [7]:
def search_narration(keyword):

    return df[
        df["narration"].str.contains(
            keyword,
            case=False,
            na=False
        )
    ].reset_index(drop=True)

In [8]:
search_narration("onion")

,narration_id,participant_id,video_id,start_timestamp,stop_timestamp,narration,verb,noun
0,P01_01_155,P01,P01_01,11:59.0,12:00.0,take cutting board with onion,take,board:cutting
1,P01_01_156,P01,P01_01,12:00.5,12:02.4,pouring onion,pour,onion
2,P01_01_76,P01,P01_01,06:21.5,06:24.9,take onion,take,onion
3,P01_01_77,P01,P01_01,06:24.0,06:24.8,take onion,take,onion
4,P01_01_78,P01,P01_01,06:24.9,06:25.6,put down onion,put-down,onion
...,...,...,...,...,...,...,...,...
1164,P28_16_3,P28,P28_16,00:00:24.94,00:00:26.17,put down onion,put-down,onion
1165,P28_16_30,P28,P28_16,00:02:04.25,00:02:15.20,continue chopping onion,chop,onion
1166,P28_16_31,P28,P28_16,00:02:10.67,00:02:18.34,continue chopping onion,chop,onion
1167,P28_16_32,P28,P28_16,00:02:18.44,00:02:41.53,continue chopping onion,chop,onion


##### Candidate Actions

In [9]:
candidate_actions = [

    ("cut", "onion"),
    ("cut", "carrot"),
    ("cut", "tomato"),

    ("chop", "onion"),
    ("slice", "bread"),
    ("peel", "potato"),

    ("stir", "pot"),
    ("stir", "pan"),
    ("stir", "saucepan"),

    ("mix", "bowl"),

    ("pour", "oil"),
    ("pour", "water"),

    ("turn-on", "hob"),
    ("turn-off", "hob"),

    ("turn-on", "oven"),
    ("turn-off", "oven"),

    ("pick-up", "knife"),
    ("put-down", "knife"),

    ("open", "fridge"),
    ("close", "fridge"),

    ("open", "cupboard"),
    ("close", "cupboard"),

    ("put-in", "pot"),
    ("take", "spoon"),

]

##### Build Candidate Dataset

In [10]:
candidate_df = pd.DataFrame()

for verb, noun in candidate_actions:

    matches = search_action(verb, noun)

    if len(matches) > 0:

        matches = matches.copy()

        matches["category"] = f"{verb} {noun}"

        candidate_df = pd.concat(
            [candidate_df, matches],
            ignore_index=True
        )

##### Remove Duplicates

In [11]:
candidate_df = candidate_df.drop_duplicates(
    subset=[
        "narration_id",
        "participant_id",
        "video_id",
        "start_timestamp",
        "stop_timestamp"
    ]
)

candidate_df.reset_index(drop=True, inplace=True)

print(f"Candidate clips: {len(candidate_df)}")

Candidate clips: 5488


##### Inspect Candidates

In [12]:
candidate_df

,narration_id,participant_id,video_id,start_timestamp,stop_timestamp,narration,verb,noun,category
0,P01_01_82,P01,P01_01,06:32.6,06:38.8,cut onion,cut,onion,cut onion
1,P01_01_83,P01,P01_01,06:35.4,06:47.0,still cutting onion,cut,onion,cut onion
2,P01_01_84,P01,P01_01,06:48.4,07:02.2,still cutting onion,cut,onion,cut onion
3,P01_09_227,P01,P01_09,18:03.9,18:10.6,cut onion,cut,onion,cut onion
4,P01_09_229,P01,P01_09,18:14.1,18:16.8,cut onion,cut,onion,cut onion
...,...,...,...,...,...,...,...,...,...
5483,P30_11_75,P30,P30_11,04:03.8,04:04.7,take spoon,take,spoon,take spoon
5484,P30_110_16,P30,P30_110,00:23.0,00:23.8,take spoon,take,spoon,take spoon
5485,P30_110_46,P30,P30_110,01:12.7,01:13.9,take spoon,take,spoon,take spoon
5486,P30_111_595,P30,P30_111,27:31.0,27:33.8,take spoon,take,spoon,take spoon


##### Count Per Category

In [13]:
candidate_df["category"].value_counts()

category
open cupboard     1199
close cupboard     798
open fridge        745
close fridge       568
pick-up knife      552
put-down knife     504
take spoon         198
cut onion          124
stir pan           121
pour water         119
turn-on hob        101
chop onion          94
cut tomato          74
turn-off hob        73
peel potato         62
pour oil            43
cut carrot          38
turn-on oven        27
turn-off oven       21
put-in pot          17
slice bread          6
mix bowl             3
stir pot             1
Name: count, dtype: int64

##### Export Candidate CSV

In [14]:
output_path = "../dataset/selected_clips/candidate_clips.csv"

candidate_df.to_csv(
    output_path,
    index=False
)

print(f"Saved to {output_path}")

Saved to ../dataset/selected_clips/candidate_clips.csv


In [15]:
candidate_df.shape

(5488, 9)

##### Export a Smaller Dataset

In [21]:
selected_df = (
    candidate_df
    .sort_values(
        ["category", "participant_id", "video_id", "start_timestamp"]
    )
    .groupby("category", group_keys=False)
    .head(10)
    .reset_index(drop=True)
)

In [22]:
selected_df.insert(
    0,
    "clip_id",
    [f"CLIP_{i:03d}" for i in range(1, len(selected_df) + 1)]
)

##### Save Final List

In [23]:
selected_path = "../dataset/selected_clips/final_selected_clips.csv"

selected_df.to_csv(
    selected_path,
    index=False
)

print(f"Selected clips: {len(selected_df)}")

Selected clips: 210


##### Display Final Dataset

In [24]:
selected_df.sort_values(
    ["category", "participant_id"]
)

,clip_id,narration_id,participant_id,video_id,start_timestamp,stop_timestamp,narration,verb,noun,category
0,CLIP_001,P01_14_143,P01,P01_14,00:07:36.89,00:07:48.80,chop onion,chop,onion,chop onion
1,CLIP_002,P01_14_144,P01,P01_14,00:07:38.37,00:07:53.77,continue chopping onion,chop,onion,chop onion
2,CLIP_003,P01_18_169,P01,P01_18,14:31.2,14:43.7,chop onion,chop,onion,chop onion
3,CLIP_004,P01_18_170,P01,P01_18,14:43.3,14:48.7,chop onions,chop,onion,chop onion
4,CLIP_005,P01_18_171,P01,P01_18,14:48.8,14:57.0,continue chopping onions,chop,onion,chop onion
...,...,...,...,...,...,...,...,...,...,...
205,CLIP_206,P03_122_4,P03,P03_122,00:14.8,00:19.2,turn on oven,turn-on,oven,turn-on oven
206,CLIP_207,P03_23_1,P03,P03_23,00:00:03.74,00:00:06.56,turn on oven,turn-on,oven,turn-on oven
207,CLIP_208,P04_110_20,P04,P04_110,01:03.8,01:05.1,turn on oven,turn-on,oven,turn-on oven
208,CLIP_209,P04_113_0,P04,P04_113,00:00.0,00:03.5,turn on oven,turn-on,oven,turn-on oven


In [25]:
(
    selected_df
    .groupby(["verb", "noun"])
    .size()
    .sort_values(ascending=False)
)

verb      noun    
chop      onion       10
close     cupboard    10
          fridge      10
cut       carrot      10
          onion       10
          tomato      10
open      cupboard    10
stir      pan         10
open      fridge      10
peel      potato      10
pick-up   knife       10
pour      oil         10
          water       10
put-down  knife       10
put-in    pot         10
turn-on   hob         10
take      spoon       10
turn-off  oven        10
          hob         10
turn-on   oven        10
slice     bread        6
mix       bowl         3
stir      pot          1
dtype: int64

#### Extract unique video_ids

In [26]:
selected_df["video_id"].nunique()

62

In [27]:
print(f"Selected clips : {len(selected_df)}")
print(f"Unique videos  : {selected_df['video_id'].nunique()}")

Selected clips : 210
Unique videos  : 62


In [28]:
video_ids = sorted(selected_df["video_id"].unique())

print(video_ids[:20])
print(len(video_ids))

['P01_01', 'P01_02', 'P01_03', 'P01_04', 'P01_05', 'P01_06', 'P01_07', 'P01_09', 'P01_10', 'P01_103', 'P01_105', 'P01_109', 'P01_14', 'P01_17', 'P01_18', 'P02_03', 'P02_09', 'P02_112', 'P02_115', 'P02_12']
62


In [29]:
video_list = ",".join(video_ids)

with open("../dataset/selected_clips/video_ids.txt", "w") as f:
    f.write(video_list)

print(video_list)

P01_01,P01_02,P01_03,P01_04,P01_05,P01_06,P01_07,P01_09,P01_10,P01_103,P01_105,P01_109,P01_14,P01_17,P01_18,P02_03,P02_09,P02_112,P02_115,P02_12,P02_120,P02_122,P02_129,P02_130,P03_06,P03_07,P03_09,P03_119,P03_120,P03_122,P03_123,P03_16,P03_19,P03_23,P03_24,P04_02,P04_05,P04_109,P04_110,P04_113,P04_116,P04_117,P04_24,P05_04,P05_09,P06_101,P06_105,P06_110,P07_10,P07_103,P08_05,P11_101,P11_104,P11_15,P13_09,P16_01,P18_03,P18_06,P22_16,P24_07,P24_09,P26_02
